In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import warnings

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

from sksurv.util import Surv
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sksurv.metrics import concordance_index_censored


project_dir = Path(r"C:\Multimodal_KIRC_Project")
processed_dir = project_dir / "03_processed_data"

counts = pd.read_csv(
    processed_dir / "TCGA_KIRC_raw_gene_count_matrix.csv",
    index_col="Case ID"
)

gene_annotation = pd.read_csv(
    processed_dir / "TCGA_KIRC_gene_annotation.csv"
)

clinical_survival = pd.read_csv(
    processed_dir / "TCGA_KIRC_clinical_survival_harmonized.csv"
)

pfi_eligible = pd.read_csv(
    processed_dir / "TCGA_KIRC_final_PFI_cohort.csv"
)

fold_assignment = pd.read_csv(
    processed_dir / "TCGA_KIRC_PFI_outer_fold_assignment.csv"
)

clinical_results = pd.read_csv(
    processed_dir / "TCGA_KIRC_PFI_clinical_outer_fold_results.csv"
)

rna_results = pd.read_csv(
    processed_dir / "TCGA_KIRC_PFI_RNA_outer_fold_results.csv"
)

print("Raw counts:", counts.shape)
print("PFI eligible:", pfi_eligible.shape)
print("Locked folds:", sorted(fold_assignment["Outer Fold"].unique()))

Raw counts: (533, 60660)
PFI eligible: (529, 34)
Locked folds: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]


In [2]:
pfi_outcome = (
    pfi_eligible[
        ["Case ID", "PFI", "PFI.time"]
    ]
    .set_index("Case ID")
    .rename(
        columns={
            "PFI": "event",
            "PFI.time": "time"
        }
    )
)

pfi_ids = pfi_outcome.index.tolist()

clinical_pfi = (
    clinical_survival
    .set_index("Case ID")
    .loc[pfi_ids]
    .copy()
)

clinical_features = clinical_pfi[
    [
        "Age at Index",
        "Tumor Grade",
        "AJCC Stage"
    ]
].copy()

clinical_features.columns = [
    "age",
    "grade",
    "stage"
]

assert clinical_features.index.equals(
    pfi_outcome.index
)

print("Multimodal clinical cohort aligned.")

Multimodal clinical cohort aligned.


In [3]:
protein_coding_ids = set(
    gene_annotation.loc[
        gene_annotation["gene_type"] == "protein_coding",
        "gene_id"
    ]
)

protein_columns = [
    gene
    for gene in counts.columns
    if gene in protein_coding_ids
]

counts_protein = (
    counts
    .loc[pfi_ids, protein_columns]
    .copy()
)

print("PFI patients:", counts_protein.shape[0])
print("Protein-coding genes:", counts_protein.shape[1])

assert counts_protein.shape[0] == 529

PFI patients: 529
Protein-coding genes: 19962


In [4]:
def preprocess_rna_train_test(
    X_train_counts,
    X_test_counts,
    min_count=10,
    min_fraction=0.20,
    top_k=250
):
    min_patients = int(
        np.ceil(
            min_fraction *
            X_train_counts.shape[0]
        )
    )

    keep_genes = (
        (X_train_counts >= min_count)
        .sum(axis=0)
        >= min_patients
    )

    selected_genes = (
        X_train_counts
        .columns[keep_genes]
    )

    X_train = (
        X_train_counts[
            selected_genes
        ].copy()
    )

    X_test = (
        X_test_counts[
            selected_genes
        ].copy()
    )

    train_library = X_train.sum(axis=1)
    test_library = X_test.sum(axis=1)

    X_train_cpm = (
        X_train
        .div(train_library, axis=0)
        * 1_000_000
    )

    X_test_cpm = (
        X_test
        .div(test_library, axis=0)
        * 1_000_000
    )

    X_train_log = np.log2(
        X_train_cpm + 1
    )

    X_test_log = np.log2(
        X_test_cpm + 1
    )

    train_variances = (
        X_train_log
        .var(axis=0)
    )

    top_genes = (
        train_variances
        .sort_values(
            ascending=False
        )
        .head(
            min(
                top_k,
                len(train_variances)
            )
        )
        .index
    )

    X_train_log = (
        X_train_log[top_genes]
    )

    X_test_log = (
        X_test_log[top_genes]
    )

    scaler = StandardScaler()

    X_train_scaled = (
        scaler.fit_transform(
            X_train_log
        )
    )

    X_test_scaled = (
        scaler.transform(
            X_test_log
        )
    )

    return {
        "X_train":
            X_train_scaled,

        "X_test":
            X_test_scaled,

        "genes":
            list(top_genes),

        "n_prevalence_genes":
            len(selected_genes)
    }

In [7]:
from sklearn.pipeline import Pipeline

In [8]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [9]:
clinical_preprocessor = ColumnTransformer(...)

In [10]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

numeric_features = ["age"]
categorical_features = ["grade", "stage"]

clinical_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            SimpleImputer(strategy="median"),
            numeric_features
        ),
        (
            "categorical",
            Pipeline([
                (
                    "imputer",
                    SimpleImputer(strategy="most_frequent")
                ),
                (
                    "onehot",
                    OneHotEncoder(
                        handle_unknown="ignore",
                        drop="first",
                        sparse_output=False
                    )
                )
            ]),
            categorical_features
        )
    ]
)

print("Clinical preprocessing object created.")

Clinical preprocessing object created.


In [11]:
# Test multimodal preprocessing on outer fold 1

fold = 1

test_ids = fold_assignment.loc[
    fold_assignment["Outer Fold"] == fold,
    "Case ID"
].tolist()

train_ids = fold_assignment.loc[
    fold_assignment["Outer Fold"] != fold,
    "Case ID"
].tolist()

# Clinical data
X_clin_train_df = clinical_features.loc[train_ids].copy()
X_clin_test_df = clinical_features.loc[test_ids].copy()

# Fit clinical preprocessing on TRAINING data only
X_clin_train = clinical_preprocessor.fit_transform(
    X_clin_train_df
)

X_clin_test = clinical_preprocessor.transform(
    X_clin_test_df
)

# RNA data
X_rna_train_counts = counts_protein.loc[train_ids].copy()
X_rna_test_counts = counts_protein.loc[test_ids].copy()

prep_rna = preprocess_rna_train_test(
    X_rna_train_counts,
    X_rna_test_counts,
    min_count=10,
    min_fraction=0.20,
    top_k=250
)

X_rna_train = prep_rna["X_train"]
X_rna_test = prep_rna["X_test"]

# Combine modalities
X_train_multi = np.hstack([
    X_clin_train,
    X_rna_train
])

X_test_multi = np.hstack([
    X_clin_test,
    X_rna_test
])

print("Outer fold:", fold)
print("Training patients:", len(train_ids))
print("Test patients:", len(test_ids))
print("Clinical training shape:", X_clin_train.shape)
print("RNA training shape:", X_rna_train.shape)
print("Multimodal training shape:", X_train_multi.shape)
print("Multimodal test shape:", X_test_multi.shape)
print(
    "Genes after RNA prevalence filter:",
    prep_rna["n_prevalence_genes"]
)

Outer fold: 1
Training patients: 423
Test patients: 106
Clinical training shape: (423, 9)
RNA training shape: (423, 250)
Multimodal training shape: (423, 259)
Multimodal test shape: (106, 259)
Genes after RNA prevalence filter: 16193


In [12]:
multimodal_outer_results = []
multimodal_predictions = []
multimodal_selected_genes = []
multimodal_tuning_results = []

for fold in sorted(
    fold_assignment["Outer Fold"].astype(int).unique()
):

    print(f"\n{'='*55}")
    print(f"Running multimodal outer fold {fold}")
    print(f"{'='*55}")

    test_ids = fold_assignment.loc[
        fold_assignment["Outer Fold"] == fold,
        "Case ID"
    ].tolist()

    train_ids = fold_assignment.loc[
        fold_assignment["Outer Fold"] != fold,
        "Case ID"
    ].tolist()

    # -------------------------------------------------
    # CLINICAL DATA
    # -------------------------------------------------
    X_clin_train_df = clinical_features.loc[
        train_ids
    ].copy()

    X_clin_test_df = clinical_features.loc[
        test_ids
    ].copy()

    X_clin_train = clinical_preprocessor.fit_transform(
        X_clin_train_df
    )

    X_clin_test = clinical_preprocessor.transform(
        X_clin_test_df
    )

    # -------------------------------------------------
    # RNA DATA
    # -------------------------------------------------
    X_rna_train_counts = counts_protein.loc[
        train_ids
    ].copy()

    X_rna_test_counts = counts_protein.loc[
        test_ids
    ].copy()

    prep_rna = preprocess_rna_train_test(
        X_rna_train_counts,
        X_rna_test_counts,
        min_count=10,
        min_fraction=0.20,
        top_k=250
    )

    X_rna_train = prep_rna["X_train"]
    X_rna_test = prep_rna["X_test"]

    # -------------------------------------------------
    # COMBINE MODALITIES
    # -------------------------------------------------
    X_train_multi = np.hstack([
        X_clin_train,
        X_rna_train
    ])

    X_test_multi = np.hstack([
        X_clin_test,
        X_rna_test
    ])

    print(
        "Clinical predictors:",
        X_clin_train.shape[1]
    )

    print(
        "RNA predictors:",
        X_rna_train.shape[1]
    )

    print(
        "Total multimodal predictors:",
        X_train_multi.shape[1]
    )

    print(
        "Genes after prevalence filter:",
        prep_rna["n_prevalence_genes"]
    )

    # -------------------------------------------------
    # SURVIVAL OUTCOME
    # -------------------------------------------------
    y_train_df = pfi_outcome.loc[
        train_ids
    ].copy()

    y_test_df = pfi_outcome.loc[
        test_ids
    ].copy()

    y_train_event = (
        y_train_df["event"]
        .astype(int)
        .to_numpy()
    )

    y_train_time = (
        y_train_df["time"]
        .astype(float)
        .to_numpy()
    )

    # -------------------------------------------------
    # INNER CV TUNING
    # -------------------------------------------------
    tuning = tune_coxnet_inner_cv(
        X_train_multi,
        y_train_event,
        y_train_time,
        l1_ratios=(0.5, 0.9, 1.0),
        n_splits=3,
        random_state=3030 + int(fold),
        n_alphas=12
    )

    best = tuning.iloc[0]

    best_l1 = float(
        best["l1_ratio"]
    )

    best_alpha = float(
        best["alpha"]
    )

    print(
        "Best l1_ratio:",
        best_l1
    )

    print(
        "Best alpha:",
        best_alpha
    )

    print(
        "Best inner C-index:",
        round(
            float(
                best["mean_inner_cindex"]
            ),
            4
        )
    )

    tuning_export = tuning.copy()
    tuning_export["Outer Fold"] = int(fold)

    multimodal_tuning_results.append(
        tuning_export
    )

    # -------------------------------------------------
    # FINAL MODEL
    # -------------------------------------------------
    y_train = Surv.from_arrays(
        event=y_train_df["event"]
        .astype(bool)
        .to_numpy(),

        time=y_train_df["time"]
        .astype(float)
        .to_numpy()
    )

    final_model = CoxnetSurvivalAnalysis(
        l1_ratio=best_l1,
        alphas=[best_alpha],
        max_iter=150000,
        tol=1e-6
    )

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")

        final_model.fit(
            X_train_multi,
            y_train
        )

    # -------------------------------------------------
    # OUTER-TEST PERFORMANCE
    # -------------------------------------------------
    risk_test = final_model.predict(
        X_test_multi
    )

    y_test = Surv.from_arrays(
        event=y_test_df["event"]
        .astype(bool)
        .to_numpy(),

        time=y_test_df["time"]
        .astype(float)
        .to_numpy()
    )

    outer_cindex = (
        concordance_index_censored(
            y_test["event"],
            y_test["time"],
            risk_test
        )[0]
    )

    # -------------------------------------------------
    # NONZERO COEFFICIENTS
    # -------------------------------------------------
    coefs = np.asarray(
        final_model.coef_
    ).ravel()

    nonzero = np.abs(
        coefs
    ) > 1e-12

    n_clinical = X_clin_train.shape[1]

    clinical_nonzero = int(
        nonzero[
            :n_clinical
        ].sum()
    )

    rna_nonzero_mask = nonzero[
        n_clinical:
    ]

    selected_rna_genes = np.asarray(
        prep_rna["genes"]
    )[rna_nonzero_mask]

    print(
        "Nonzero clinical coefficients:",
        clinical_nonzero
    )

    print(
        "Nonzero RNA genes:",
        len(
            selected_rna_genes
        )
    )

    print(
        "Outer-test C-index:",
        round(
            float(
                outer_cindex
            ),
            4
        )
    )

    # -------------------------------------------------
    # STORE FOLD RESULTS
    # -------------------------------------------------
    multimodal_outer_results.append({
        "Fold":
            int(fold),

        "Train N":
            len(train_ids),

        "Test N":
            len(test_ids),

        "Test Events":
            int(
                y_test_df[
                    "event"
                ].sum()
            ),

        "Clinical predictors":
            int(
                X_clin_train.shape[1]
            ),

        "RNA genes":
            int(
                X_rna_train.shape[1]
            ),

        "Total predictors":
            int(
                X_train_multi.shape[1]
            ),

        "Nonzero clinical":
            clinical_nonzero,

        "Nonzero RNA genes":
            int(
                len(
                    selected_rna_genes
                )
            ),

        "Best l1_ratio":
            best_l1,

        "Best alpha":
            best_alpha,

        "Inner CV C-index":
            float(
                best[
                    "mean_inner_cindex"
                ]
            ),

        "Outer C-index":
            float(
                outer_cindex
            )
    })

    # -------------------------------------------------
    # STORE SELECTED RNA GENES
    # -------------------------------------------------
    for gene in selected_rna_genes:

        multimodal_selected_genes.append({
            "Fold":
                int(fold),

            "gene_id":
                gene
        })

    # -------------------------------------------------
    # STORE OUT-OF-FOLD PREDICTIONS
    # -------------------------------------------------
    for case_id, score in zip(
        test_ids,
        risk_test
    ):

        multimodal_predictions.append({
            "Case ID":
                case_id,

            "Fold":
                int(fold),

            "PFI_event":
                int(
                    y_test_df.loc[
                        case_id,
                        "event"
                    ]
                ),

            "PFI_time":
                float(
                    y_test_df.loc[
                        case_id,
                        "time"
                    ]
                ),

            "Multimodal_Risk":
                float(score)
        })


# =====================================================
# CONVERT RESULTS
# =====================================================

multimodal_outer_results = pd.DataFrame(
    multimodal_outer_results
)

multimodal_predictions = pd.DataFrame(
    multimodal_predictions
)

multimodal_selected_genes = pd.DataFrame(
    multimodal_selected_genes
)

multimodal_tuning_results = pd.concat(
    multimodal_tuning_results,
    ignore_index=True
)

display(
    multimodal_outer_results
)

multimodal_mean_cindex = (
    multimodal_outer_results[
        "Outer C-index"
    ].mean()
)

multimodal_sd_cindex = (
    multimodal_outer_results[
        "Outer C-index"
    ].std(ddof=1)
)

print(
    "\nMultimodal mean outer-fold C-index:",
    multimodal_mean_cindex
)

print(
    "Multimodal SD outer-fold C-index:",
    multimodal_sd_cindex
)

print(
    "\nOut-of-fold predictions:",
    len(
        multimodal_predictions
    )
)

print(
    "Unique predicted patients:",
    multimodal_predictions[
        "Case ID"
    ].nunique()
)

assert len(
    multimodal_outer_results
) == 5

assert len(
    multimodal_predictions
) == 529

assert multimodal_predictions[
    "Case ID"
].nunique() == 529

print(
    "\nAll multimodal outer-CV integrity checks passed."
)


Running multimodal outer fold 1
Clinical predictors: 9
RNA predictors: 250
Total multimodal predictors: 259
Genes after prevalence filter: 16193


NameError: name 'tune_coxnet_inner_cv' is not defined

In [13]:
from sklearn.model_selection import StratifiedKFold
from sksurv.util import Surv
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sksurv.metrics import concordance_index_censored

import warnings
import numpy as np
import pandas as pd


def tune_coxnet_inner_cv(
    X,
    y_event,
    y_time,
    l1_ratios=(0.5, 0.9, 1.0),
    n_splits=3,
    random_state=2026,
    n_alphas=12
):

    inner_cv = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state
    )

    tuning_rows = []

    for l1_ratio in l1_ratios:

        y_outer_train = Surv.from_arrays(
            event=y_event.astype(bool),
            time=y_time.astype(float)
        )

        path_model = CoxnetSurvivalAnalysis(
            l1_ratio=l1_ratio,
            n_alphas=n_alphas,
            alpha_min_ratio=0.01,
            max_iter=150000,
            tol=1e-6
        )

        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")

                path_model.fit(
                    X,
                    y_outer_train
                )

        except Exception:
            continue

        candidate_alphas = path_model.alphas_

        for alpha in candidate_alphas:

            fold_scores = []

            for train_idx, val_idx in inner_cv.split(
                X,
                y_event
            ):

                y_train = Surv.from_arrays(
                    event=y_event[
                        train_idx
                    ].astype(bool),

                    time=y_time[
                        train_idx
                    ].astype(float)
                )

                y_val = Surv.from_arrays(
                    event=y_event[
                        val_idx
                    ].astype(bool),

                    time=y_time[
                        val_idx
                    ].astype(float)
                )

                model = CoxnetSurvivalAnalysis(
                    l1_ratio=l1_ratio,
                    alphas=[alpha],
                    max_iter=150000,
                    tol=1e-6
                )

                try:
                    with warnings.catch_warnings():
                        warnings.simplefilter("ignore")

                        model.fit(
                            X[train_idx],
                            y_train
                        )

                    coef = np.asarray(
                        model.coef_
                    ).ravel()

                    if np.all(
                        np.abs(coef) < 1e-12
                    ):
                        continue

                    risk = model.predict(
                        X[val_idx]
                    )

                    if not np.isfinite(
                        risk
                    ).all():
                        continue

                    cindex = (
                        concordance_index_censored(
                            y_val["event"],
                            y_val["time"],
                            risk
                        )[0]
                    )

                    if np.isfinite(cindex):
                        fold_scores.append(
                            float(cindex)
                        )

                except Exception:
                    continue

            if len(fold_scores) >= 3:

                tuning_rows.append({
                    "l1_ratio":
                        float(l1_ratio),

                    "alpha":
                        float(alpha),

                    "valid_inner_folds":
                        len(fold_scores),

                    "mean_inner_cindex":
                        float(
                            np.mean(
                                fold_scores
                            )
                        ),

                    "sd_inner_cindex":
                        float(
                            np.std(
                                fold_scores,
                                ddof=1
                            )
                        )
                })

    tuning_df = pd.DataFrame(
        tuning_rows
    )

    if tuning_df.empty:
        raise RuntimeError(
            "No valid Coxnet hyperparameter combinations were found."
        )

    tuning_df = (
        tuning_df
        .sort_values(
            [
                "mean_inner_cindex",
                "sd_inner_cindex"
            ],
            ascending=[
                False,
                True
            ]
        )
        .reset_index(
            drop=True
        )
    )

    return tuning_df


print("Fast Coxnet tuning function loaded successfully.")

Fast Coxnet tuning function loaded successfully.


In [14]:
multimodal_outer_results = []
multimodal_predictions = []
multimodal_selected_genes = []
multimodal_tuning_results = []

for fold in sorted(
    fold_assignment["Outer Fold"].astype(int).unique()
):

    print(f"\n{'='*55}")
    print(f"Running multimodal outer fold {fold}")
    print(f"{'='*55}")

    # -------------------------------------------------
    # Locked outer train/test IDs
    # -------------------------------------------------
    test_ids = fold_assignment.loc[
        fold_assignment["Outer Fold"] == fold,
        "Case ID"
    ].tolist()

    train_ids = fold_assignment.loc[
        fold_assignment["Outer Fold"] != fold,
        "Case ID"
    ].tolist()

    # -------------------------------------------------
    # Clinical data
    # -------------------------------------------------
    X_clin_train_df = clinical_features.loc[
        train_ids
    ].copy()

    X_clin_test_df = clinical_features.loc[
        test_ids
    ].copy()

    X_clin_train = clinical_preprocessor.fit_transform(
        X_clin_train_df
    )

    X_clin_test = clinical_preprocessor.transform(
        X_clin_test_df
    )

    # -------------------------------------------------
    # RNA data
    # -------------------------------------------------
    X_rna_train_counts = counts_protein.loc[
        train_ids
    ].copy()

    X_rna_test_counts = counts_protein.loc[
        test_ids
    ].copy()

    prep_rna = preprocess_rna_train_test(
        X_rna_train_counts,
        X_rna_test_counts,
        min_count=10,
        min_fraction=0.20,
        top_k=250
    )

    X_rna_train = prep_rna["X_train"]
    X_rna_test = prep_rna["X_test"]

    # -------------------------------------------------
    # Combine clinical + RNA features
    # -------------------------------------------------
    X_train_multi = np.hstack([
        X_clin_train,
        X_rna_train
    ])

    X_test_multi = np.hstack([
        X_clin_test,
        X_rna_test
    ])

    print(
        "Clinical predictors:",
        X_clin_train.shape[1]
    )

    print(
        "RNA predictors:",
        X_rna_train.shape[1]
    )

    print(
        "Total multimodal predictors:",
        X_train_multi.shape[1]
    )

    print(
        "Genes after prevalence filter:",
        prep_rna["n_prevalence_genes"]
    )

    # -------------------------------------------------
    # Survival outcome
    # -------------------------------------------------
    y_train_df = pfi_outcome.loc[
        train_ids
    ].copy()

    y_test_df = pfi_outcome.loc[
        test_ids
    ].copy()

    y_train_event = (
        y_train_df["event"]
        .astype(int)
        .to_numpy()
    )

    y_train_time = (
        y_train_df["time"]
        .astype(float)
        .to_numpy()
    )

    # -------------------------------------------------
    # Inner CV tuning
    # -------------------------------------------------
    tuning = tune_coxnet_inner_cv(
        X_train_multi,
        y_train_event,
        y_train_time,
        l1_ratios=(0.5, 0.9, 1.0),
        n_splits=3,
        random_state=3030 + int(fold),
        n_alphas=12
    )

    best = tuning.iloc[0]

    best_l1 = float(
        best["l1_ratio"]
    )

    best_alpha = float(
        best["alpha"]
    )

    print(
        "Best l1_ratio:",
        best_l1
    )

    print(
        "Best alpha:",
        best_alpha
    )

    print(
        "Best inner C-index:",
        round(
            float(
                best["mean_inner_cindex"]
            ),
            4
        )
    )

    # Save tuning results
    tuning_export = tuning.copy()
    tuning_export["Outer Fold"] = int(fold)

    multimodal_tuning_results.append(
        tuning_export
    )

    # -------------------------------------------------
    # Final model on complete outer-training set
    # -------------------------------------------------
    y_train = Surv.from_arrays(
        event=y_train_df["event"]
        .astype(bool)
        .to_numpy(),

        time=y_train_df["time"]
        .astype(float)
        .to_numpy()
    )

    final_model = CoxnetSurvivalAnalysis(
        l1_ratio=best_l1,
        alphas=[best_alpha],
        max_iter=150000,
        tol=1e-6
    )

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")

        final_model.fit(
            X_train_multi,
            y_train
        )

    # -------------------------------------------------
    # Predict on untouched outer-test patients
    # -------------------------------------------------
    risk_test = final_model.predict(
        X_test_multi
    )

    y_test = Surv.from_arrays(
        event=y_test_df["event"]
        .astype(bool)
        .to_numpy(),

        time=y_test_df["time"]
        .astype(float)
        .to_numpy()
    )

    outer_cindex = (
        concordance_index_censored(
            y_test["event"],
            y_test["time"],
            risk_test
        )[0]
    )

    # -------------------------------------------------
    # Inspect nonzero coefficients
    # -------------------------------------------------
    coefs = np.asarray(
        final_model.coef_
    ).ravel()

    nonzero = (
        np.abs(coefs) > 1e-12
    )

    n_clinical = X_clin_train.shape[1]

    clinical_nonzero = int(
        nonzero[:n_clinical].sum()
    )

    rna_nonzero_mask = (
        nonzero[n_clinical:]
    )

    selected_rna_genes = np.asarray(
        prep_rna["genes"]
    )[rna_nonzero_mask]

    print(
        "Nonzero clinical coefficients:",
        clinical_nonzero
    )

    print(
        "Nonzero RNA genes:",
        len(selected_rna_genes)
    )

    print(
        "Outer-test C-index:",
        round(
            float(outer_cindex),
            4
        )
    )

    # -------------------------------------------------
    # Store fold-level results
    # -------------------------------------------------
    multimodal_outer_results.append({
        "Fold":
            int(fold),

        "Train N":
            len(train_ids),

        "Test N":
            len(test_ids),

        "Test Events":
            int(
                y_test_df[
                    "event"
                ].sum()
            ),

        "Clinical predictors":
            int(
                X_clin_train.shape[1]
            ),

        "RNA genes":
            int(
                X_rna_train.shape[1]
            ),

        "Total predictors":
            int(
                X_train_multi.shape[1]
            ),

        "Nonzero clinical":
            clinical_nonzero,

        "Nonzero RNA genes":
            int(
                len(selected_rna_genes)
            ),

        "Best l1_ratio":
            best_l1,

        "Best alpha":
            best_alpha,

        "Inner CV C-index":
            float(
                best[
                    "mean_inner_cindex"
                ]
            ),

        "Outer C-index":
            float(
                outer_cindex
            )
    })

    # -------------------------------------------------
    # Store selected RNA genes
    # -------------------------------------------------
    for gene in selected_rna_genes:

        multimodal_selected_genes.append({
            "Fold": int(fold),
            "gene_id": gene
        })

    # -------------------------------------------------
    # Store held-out patient predictions
    # -------------------------------------------------
    for case_id, score in zip(
        test_ids,
        risk_test
    ):

        multimodal_predictions.append({
            "Case ID":
                case_id,

            "Fold":
                int(fold),

            "PFI_event":
                int(
                    y_test_df.loc[
                        case_id,
                        "event"
                    ]
                ),

            "PFI_time":
                float(
                    y_test_df.loc[
                        case_id,
                        "time"
                    ]
                ),

            "Multimodal_Risk":
                float(score)
        })


# =====================================================
# Convert outputs after all 5 outer folds
# =====================================================

multimodal_outer_results = pd.DataFrame(
    multimodal_outer_results
)

multimodal_predictions = pd.DataFrame(
    multimodal_predictions
)

multimodal_selected_genes = pd.DataFrame(
    multimodal_selected_genes
)

multimodal_tuning_results = pd.concat(
    multimodal_tuning_results,
    ignore_index=True
)

# -----------------------------------------------------
# Display results
# -----------------------------------------------------
display(
    multimodal_outer_results
)

multimodal_mean_cindex = (
    multimodal_outer_results[
        "Outer C-index"
    ].mean()
)

multimodal_sd_cindex = (
    multimodal_outer_results[
        "Outer C-index"
    ].std(ddof=1)
)

print(
    "\nMultimodal mean outer-fold C-index:",
    multimodal_mean_cindex
)

print(
    "Multimodal SD outer-fold C-index:",
    multimodal_sd_cindex
)

print(
    "\nOut-of-fold predictions:",
    len(multimodal_predictions)
)

print(
    "Unique predicted patients:",
    multimodal_predictions[
        "Case ID"
    ].nunique()
)

# -----------------------------------------------------
# Integrity checks
# -----------------------------------------------------
assert len(multimodal_outer_results) == 5
assert len(multimodal_predictions) == 529
assert (
    multimodal_predictions[
        "Case ID"
    ].nunique()
    == 529
)

print(
    "\nAll multimodal outer-CV integrity checks passed."
)


Running multimodal outer fold 1
Clinical predictors: 9
RNA predictors: 250
Total multimodal predictors: 259
Genes after prevalence filter: 16193
Best l1_ratio: 1.0
Best alpha: 0.03930236997692589
Best inner C-index: 0.7765
Nonzero clinical coefficients: 3
Nonzero RNA genes: 25
Outer-test C-index: 0.8065

Running multimodal outer fold 2
Clinical predictors: 9
RNA predictors: 250
Total multimodal predictors: 259
Genes after prevalence filter: 16185
Best l1_ratio: 1.0
Best alpha: 0.021567834249007347
Best inner C-index: 0.7921
Nonzero clinical coefficients: 3
Nonzero RNA genes: 53
Outer-test C-index: 0.8052

Running multimodal outer fold 3


C:\Users\kumia\miniforge3\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


Clinical predictors: 8
RNA predictors: 250
Total multimodal predictors: 258
Genes after prevalence filter: 16175
Best l1_ratio: 1.0
Best alpha: 0.05154535899833902
Best inner C-index: 0.8066
Nonzero clinical coefficients: 3
Nonzero RNA genes: 20
Outer-test C-index: 0.7883

Running multimodal outer fold 4
Clinical predictors: 9
RNA predictors: 250
Total multimodal predictors: 259
Genes after prevalence filter: 16176
Best l1_ratio: 1.0
Best alpha: 0.041003560406695574
Best inner C-index: 0.7965
Nonzero clinical coefficients: 2
Nonzero RNA genes: 25
Outer-test C-index: 0.7765

Running multimodal outer fold 5
Clinical predictors: 9
RNA predictors: 250
Total multimodal predictors: 259
Genes after prevalence filter: 16193
Best l1_ratio: 1.0
Best alpha: 0.01570882922607043
Best inner C-index: 0.7898
Nonzero clinical coefficients: 5
Nonzero RNA genes: 57
Outer-test C-index: 0.7656


,Fold,Train N,Test N,Test Events,Clinical predictors,RNA genes,Total predictors,Nonzero clinical,Nonzero RNA genes,Best l1_ratio,Best alpha,Inner CV C-index,Outer C-index
0,1,423,106,32,9,250,259,3,25,1.0,0.039302,0.776503,0.806529
1,2,423,106,32,9,250,259,3,53,1.0,0.021568,0.792103,0.805206
2,3,423,106,32,8,250,258,3,20,1.0,0.051545,0.806622,0.788274
3,4,423,106,32,9,250,259,2,25,1.0,0.041004,0.796476,0.776517
4,5,424,105,31,9,250,259,5,57,1.0,0.015709,0.789835,0.765618



Multimodal mean outer-fold C-index: 0.7884286827961458
Multimodal SD outer-fold C-index: 0.017828024128217563

Out-of-fold predictions: 529
Unique predicted patients: 529

All multimodal outer-CV integrity checks passed.


In [15]:
multimodal_outer_results.to_csv(
    processed_dir / "TCGA_KIRC_PFI_multimodal_outer_fold_results.csv",
    index=False
)

multimodal_predictions.to_csv(
    processed_dir / "TCGA_KIRC_PFI_multimodal_out_of_fold_predictions.csv",
    index=False
)

multimodal_selected_genes.to_csv(
    processed_dir / "TCGA_KIRC_PFI_multimodal_selected_genes.csv",
    index=False
)

multimodal_tuning_results.to_csv(
    processed_dir / "TCGA_KIRC_PFI_multimodal_inner_tuning_results.csv",
    index=False
)

print("Multimodal results saved.")

Multimodal results saved.


In [16]:
model_comparison = pd.DataFrame({
    "Fold": clinical_results["Fold"],
    "Clinical": clinical_results["C-index"],
    "RNA": rna_results["Outer C-index"],
    "Clinical + RNA": multimodal_outer_results["Outer C-index"]
})

model_comparison["RNA - Clinical"] = (
    model_comparison["RNA"]
    - model_comparison["Clinical"]
)

model_comparison["Multimodal - Clinical"] = (
    model_comparison["Clinical + RNA"]
    - model_comparison["Clinical"]
)

model_comparison["Multimodal - RNA"] = (
    model_comparison["Clinical + RNA"]
    - model_comparison["RNA"]
)

display(model_comparison)

print("\nMean performance")
print("Clinical:", model_comparison["Clinical"].mean())
print("RNA:", model_comparison["RNA"].mean())
print("Multimodal:", model_comparison["Clinical + RNA"].mean())

print(
    "\nMean multimodal improvement over RNA:",
    model_comparison["Multimodal - RNA"].mean()
)

print(
    "Mean multimodal improvement over clinical:",
    model_comparison["Multimodal - Clinical"].mean()
)

model_comparison.to_csv(
    processed_dir / "TCGA_KIRC_PFI_three_model_comparison.csv",
    index=False
)

,Fold,Clinical,RNA,Clinical + RNA,RNA - Clinical,Multimodal - Clinical,Multimodal - RNA
0,1,0.843975,0.710034,0.806529,-0.133941,-0.037446,0.096495
1,2,0.842516,0.760521,0.805206,-0.081996,-0.037310,0.044685
2,3,0.733364,0.705444,0.788274,-0.027920,0.054909,0.082829
3,4,0.843736,0.645133,0.776517,-0.198603,-0.067220,0.131384
4,5,0.801642,0.674875,0.765618,-0.126767,-0.036024,0.090743



Mean performance
Clinical: 0.8130467214072006
RNA: 0.6992012672267691
Multimodal: 0.7884286827961458

Mean multimodal improvement over RNA: 0.08922741556937655
Mean multimodal improvement over clinical: -0.02461803861105487


In [17]:
# Count how often each RNA gene was selected across outer folds

gene_stability = (
    multimodal_selected_genes
    .groupby("gene_id")
    .agg(
        Selection_Frequency=("Fold", "nunique")
    )
    .reset_index()
    .sort_values(
        ["Selection_Frequency", "gene_id"],
        ascending=[False, True]
    )
)

display(gene_stability.head(30))

print(
    "Unique RNA genes selected across all folds:",
    gene_stability.shape[0]
)

print("\nSelection-frequency distribution:")
display(
    gene_stability["Selection_Frequency"]
    .value_counts()
    .sort_index()
    .rename_axis("Number of folds selected")
    .reset_index(name="Number of genes")
)

,gene_id,Selection_Frequency
32,ENSG00000125255.7,5
52,ENSG00000147003.7,5
64,ENSG00000163735.7,5
73,ENSG00000173376.14,5
75,ENSG00000174358.16,5
1,ENSG00000006747.15,4
10,ENSG00000077274.9,4
12,ENSG00000087250.9,4
38,ENSG00000131482.10,4
63,ENSG00000163581.14,4


Unique RNA genes selected across all folds: 94

Selection-frequency distribution:


,Number of folds selected,Number of genes
0,1,46
1,2,27
2,3,9
3,4,7
4,5,5


In [18]:
from itertools import combinations

fold_gene_sets = {
    fold: set(
        multimodal_selected_genes.loc[
            multimodal_selected_genes["Fold"] == fold,
            "gene_id"
        ]
    )
    for fold in sorted(
        multimodal_selected_genes["Fold"].unique()
    )
}

jaccard_rows = []

for fold_a, fold_b in combinations(
    sorted(fold_gene_sets.keys()),
    2
):
    genes_a = fold_gene_sets[fold_a]
    genes_b = fold_gene_sets[fold_b]

    intersection = len(
        genes_a.intersection(genes_b)
    )

    union = len(
        genes_a.union(genes_b)
    )

    jaccard = (
        intersection / union
        if union > 0
        else np.nan
    )

    jaccard_rows.append({
        "Fold A": fold_a,
        "Fold B": fold_b,
        "Genes A": len(genes_a),
        "Genes B": len(genes_b),
        "Shared genes": intersection,
        "Jaccard index": jaccard
    })

jaccard_df = pd.DataFrame(jaccard_rows)

display(jaccard_df)

print(
    "\nMean pairwise Jaccard index:",
    jaccard_df["Jaccard index"].mean()
)

,Fold A,Fold B,Genes A,Genes B,Shared genes,Jaccard index
0,1,2,25,53,14,0.218750
1,1,3,25,20,11,0.323529
2,1,4,25,25,14,0.388889
3,1,5,25,57,15,0.223881
4,2,3,53,20,10,0.158730
5,2,4,53,25,16,0.258065
6,2,5,53,57,31,0.392405
7,3,4,20,25,13,0.406250
8,3,5,20,57,10,0.149254
9,4,5,25,57,12,0.171429



Mean pairwise Jaccard index: 0.26911809385907054


In [19]:
stable_genes = gene_stability[
    gene_stability["Selection_Frequency"] >= 3
].copy()

print(
    "Genes selected in at least 3 of 5 folds:",
    len(stable_genes)
)

display(stable_genes)

Genes selected in at least 3 of 5 folds: 21


,gene_id,Selection_Frequency
32,ENSG00000125255.7,5
52,ENSG00000147003.7,5
64,ENSG00000163735.7,5
73,ENSG00000173376.14,5
75,ENSG00000174358.16,5
1,ENSG00000006747.15,4
10,ENSG00000077274.9,4
12,ENSG00000087250.9,4
38,ENSG00000131482.10,4
63,ENSG00000163581.14,4


In [20]:
stable_genes_annotated = stable_genes.merge(
    gene_annotation[
        ["gene_id", "gene_name", "gene_type"]
    ],
    on="gene_id",
    how="left"
)

stable_genes_annotated = (
    stable_genes_annotated
    .sort_values(
        ["Selection_Frequency", "gene_name"],
        ascending=[False, True]
    )
)

display(stable_genes_annotated)

stable_genes_annotated.to_csv(
    processed_dir / "TCGA_KIRC_PFI_stable_multimodal_RNA_genes.csv",
    index=False
)

,gene_id,Selection_Frequency,gene_name,gene_type
1,ENSG00000147003.7,5,CLTRN,protein_coding
2,ENSG00000163735.7,5,CXCL5,protein_coding
3,ENSG00000173376.14,5,NDNF,protein_coding
0,ENSG00000125255.7,5,SLC10A2,protein_coding
4,ENSG00000174358.16,5,SLC6A19,protein_coding
6,ENSG00000077274.9,4,CAPN6,protein_coding
10,ENSG00000164434.12,4,FABP7,protein_coding
8,ENSG00000131482.10,4,G6PC,protein_coding
7,ENSG00000087250.9,4,MT3,protein_coding
11,ENSG00000185686.18,4,PRAME,protein_coding


In [21]:
gene_stability_annotated = gene_stability.merge(
    gene_annotation[
        ["gene_id", "gene_name", "gene_type"]
    ],
    on="gene_id",
    how="left"
)

display(
    gene_stability_annotated.head(30)
)

gene_stability_annotated.to_csv(
    processed_dir / "TCGA_KIRC_PFI_multimodal_gene_selection_stability.csv",
    index=False
)

,gene_id,Selection_Frequency,gene_name,gene_type
0,ENSG00000125255.7,5,SLC10A2,protein_coding
1,ENSG00000147003.7,5,CLTRN,protein_coding
2,ENSG00000163735.7,5,CXCL5,protein_coding
3,ENSG00000173376.14,5,NDNF,protein_coding
4,ENSG00000174358.16,5,SLC6A19,protein_coding
5,ENSG00000006747.15,4,SCIN,protein_coding
6,ENSG00000077274.9,4,CAPN6,protein_coding
7,ENSG00000087250.9,4,MT3,protein_coding
8,ENSG00000131482.10,4,G6PC,protein_coding
9,ENSG00000163581.14,4,SLC2A2,protein_coding


In [22]:
jaccard_df.to_csv(
    processed_dir / "TCGA_KIRC_PFI_multimodal_gene_jaccard.csv",
    index=False
)

print("Feature-stability results saved.")

Feature-stability results saved.
